In [8]:
# loading data

from torchvision import datasets, transforms
from torch import nn

from torch.utils.data import Dataset, DataLoader
from torch import optim
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [9]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform= transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

BATCH = 64

train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH, shuffle=False)

In [10]:
def init_cnn(module):
    # initialize weights 
    if type(module) == nn.LazyLinear or type(module) == nn.LazyConv2d:
        nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')

class LeNet(nn.Module):
    def __init__(self, lr = 0.1, num_classes = 10):
        super().__init__()
        self.lr = lr
        self.num_classes = num_classes

        self.net = nn.Sequential(
            nn.LazyConv2d(8, kernel_size=3, padding  = 1), nn.ReLU(), # (64, 8, 28 , 28) 
            nn.MaxPool2d(kernel_size=2, stride = 2), # halves dimensions (64, 8, 14, 14)
            nn.LazyConv2d(16, kernel_size=3, padding = 1), nn.ReLU(), # (64, 16, 14, 14)
            nn.LazyConv2d(32, kernel_size=3, padding = 1), nn.ReLU(), # (64, 32, 14, 14)
            # now linear part
            nn.Flatten(), # (6272,)
            nn.LazyLinear(1568), nn.ReLU(), # 4x decrease
            nn.LazyLinear(256), nn.ReLU(),
            nn.LazyLinear(num_classes)

        )
    
    def forward(self, X):
        return self.net(X)

In [11]:
def initialize_obj(lr=0.1):
    model = LeNet().to(device)
    model(torch.zeros(1, 1, 28, 28).to(device)) 
    model.apply(init_cnn)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    return model, optimizer


def trainer(model, optimizer):

    model.train()
    
    loss = nn.CrossEntropyLoss()

    for batch_img, batch_label in train_loader:
        batch_img, batch_label = batch_img.to(device), batch_label.to(device)
        y_hat = model.forward(batch_img)
        output = loss(y_hat, batch_label)
        optimizer.zero_grad()
        output.backward()
        optimizer.step()


In [12]:
model, optimizer = initialize_obj(0.1)

MAX_EPOCH = 10

for _ in range(MAX_EPOCH):
    trainer(model, optimizer)

In [13]:
def eval(model):
    model.eval()
    batch_correct = []
    with torch.no_grad():
        for batch_img, batch_label in test_loader:
            batch_img, batch_label = batch_img.to(device), batch_label.to(device)
            y_hat = model.forward(batch_img)
            predicted = torch.argmax(y_hat, dim=1)
            
            results = predicted == batch_label
            correct = sum(results)
            batch_correct.append(correct/len(batch_label))
    
    return sum(batch_correct)/len(batch_correct)



In [17]:
accuracy = eval(model)
print(f'Achieved {float(accuracy)*100}% accuracy.')

Achieved 98.93510937690735% accuracy.
